# Meth3D-Net V6 — TCGA Q-Arm Only Probe Extractor
## Extracts probes overlapping V6 q-arm DMBs only

### Why q-arm only?
- Q-arms contain the majority of significant DMBs (chr10_q: 7,026 DMBs vs chr10_p: 3,273)
- P-arms of acrocentric chromosomes (chr13/14/15/21/22) are nearly empty
- Reduces output file size while retaining the most biologically informative regions
- Separate from the full-genome file — upload as a second Kaggle dataset for comparison

**Output:** `tcga_dmb_probes_q_only.tsv` — smaller, q-arm DMBs only
**Upload to Kaggle:** Add to `tcga-pancan-methylation` dataset alongside the full file


## Step 0 — Mount Drive and configure paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_PATH   = '/content/drive/MyDrive/Meth3DNet_TCGA'
ROOT_DIR    = f'{BASE_PATH}/Methylation_Paper_CpG_v6'
SUB_DIR     = f'{ROOT_DIR}/Methylation Paper cpG_v6'
METH_FILE   = f'{BASE_PATH}/jhu-usc.edu_PANCAN_HumanMethylation450.betaValue.tsv'
MANIFEST    = f'{BASE_PATH}/humanmethylation450_15017482_v1-2.csv'
OUTPUT_FILE = f'{BASE_PATH}/tcga_dmb_probes_q_only.tsv'
WINDOW_SIZE = 50_000

print('Paths configured:')
for name, path in [('Root DMB dir', ROOT_DIR), ('Sub DMB dir', SUB_DIR),
                    ('TCGA file', METH_FILE), ('Manifest', MANIFEST),
                    ('Output', OUTPUT_FILE)]:
    exists = os.path.exists(path)
    sz = ''
    if exists and os.path.isfile(path):
        gb = os.path.getsize(path)/1e9
        sz = f'  ({gb:.1f} GB)' if gb > 0.1 else f'  ({os.path.getsize(path)/1e6:.0f} MB)'
    print(f'  {"OK" if exists else "MISSING"}: {name}{sz}')


## Step 1 — Load V6 Q-arm DMBs only
Scans both directories but **only loads `_V6_dmb_q.csv` files**.
P-arm files are explicitly skipped.

In [ ]:
import numpy as np
import pandas as pd

CHROMS = [f'chr{i}' for i in list(range(1,23))+['X','Y']]

print('Loading V6 Q-arm DMBs only...')
all_dmbs, seen = [], set()
skipped_p = []

for search_dir in [ROOT_DIR, SUB_DIR]:
    if not os.path.exists(search_dir):
        print(f'  Not found: {search_dir}')
        continue
    for fname in sorted(os.listdir(search_dir)):
        if '_V6_dmb_' not in fname or not fname.endswith('.csv'):
            continue
        if fname in seen:
            continue
        seen.add(fname)

        # ── SKIP ALL P-ARM FILES ──────────────────────────
        if '_V6_dmb_p' in fname:
            skipped_p.append(fname)
            continue

        # Only q-arm files reach here
        fpath = os.path.join(search_dir, fname)
        try:
            if os.path.getsize(fpath) <= 1: continue
            df = pd.read_csv(fpath)
            if len(df) == 0: continue
            chrom = fname.split('_')[0]
            df['chrom'] = chrom
            df['arm']   = 'q'
            if 'abs_delta' not in df.columns and 'delta' in df.columns:
                df['abs_delta'] = df['delta'].abs()
            if 'ram_sig_99' not in df.columns:
                df['ram_sig_99'] = df['abs_delta'] > 0.20
            if 'start' not in df.columns and 'mid_mb' in df.columns:
                df['start'] = (df['mid_mb']*1e6).astype(int)
                df['end']   = df['start'] + WINDOW_SIZE
            sig = df[df['ram_sig_99']]
            if len(sig):
                all_dmbs.append(sig)
                print(f'  Loaded {fname}: {len(sig):,} sig DMBs')
        except (pd.errors.EmptyDataError, pd.errors.ParserError):
            pass
        except Exception as e:
            print(f'  Skip {fname}: {e}')

print(f'\nP-arm files skipped: {len(skipped_p)}')
for f in sorted(skipped_p): print(f'  {f}')

if not all_dmbs:
    raise RuntimeError('No q-arm DMB files found. Check paths.')

dmb_df = pd.concat(all_dmbs, ignore_index=True)
print(f'\nQ-arm DMBs (significant): {len(dmb_df):,}')
print(f'Chromosomes             : {sorted(dmb_df.chrom.unique())}')
print(f'Mean |Δβ|               : {dmb_df["abs_delta"].mean():.4f}')


## Step 2 — Build q-arm probe set from Illumina 450K manifest

In [ ]:
print('Building q-arm probe set from manifest...')
TARGET_PROBES = set()

header_row = 0
with open(MANIFEST, 'r', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        if line.startswith('IlmnID') or line.startswith('Name'):
            header_row = i; break
print(f'Manifest header at row {header_row}')

man = pd.read_csv(MANIFEST, skiprows=header_row, low_memory=False)
name_col = next((c for c in man.columns if c in ['Name','IlmnID']), man.columns[0])
chr_col  = next((c for c in man.columns if c in ['CHR','Chr','Chromosome']), None)
pos_col  = next((c for c in man.columns if c in ['MAPINFO','MapInfo','Position']), None)
print(f'Manifest: {man.shape}  probe={name_col} chr={chr_col} pos={pos_col}')

man = man[[name_col, chr_col, pos_col]].dropna()
man.columns = ['probe_id','chrom','position']
man['chrom']    = 'chr' + man['chrom'].astype(str).str.replace('chr','',regex=False)
man['position'] = pd.to_numeric(man['position'], errors='coerce')
man = man.dropna(subset=['position'])
man['position'] = man['position'].astype(int)
print(f'Valid probes: {len(man):,}')

print('Intersecting with V6 Q-arm DMB windows...')
for chrom in CHROMS:
    intervals = dmb_df[dmb_df['chrom']==chrom].sort_values('start')
    probes    = man[man['chrom']==chrom]
    if len(intervals)==0 or len(probes)==0: continue
    starts = intervals['start'].values
    ends   = intervals['end'].values
    for _, row in probes.iterrows():
        pos = row['position']
        idx = np.searchsorted(starts, pos, side='right') - 1
        if idx >= 0 and pos <= ends[idx]:
            TARGET_PROBES.add(row['probe_id'])

print(f'\nProbes overlapping V6 Q-arm DMBs: {len(TARGET_PROBES):,}')
print(f'(vs 134,178 for full p+q — q-arm only is a focused subset)')


## Step 3 — Scan TCGA methylation file (low-RAM, writes to disk)
**Takes 20–60 minutes.** Writes each matching probe directly to disk — no RAM accumulation.

In [ ]:
print('Scanning TCGA file (q-arm probes only, low-RAM mode)...')
print(f'File: {METH_FILE}')
print(f'Size: {os.path.getsize(METH_FILE)/1e9:.1f} GB')
print(f'Target probes: {len(TARGET_PROBES):,}')

total_lines = 0
found_count = 0

with open(METH_FILE, 'rt', errors='replace') as fh, \
     open(OUTPUT_FILE, 'wt', encoding='utf-8') as out:

    header = fh.readline().rstrip('\n')
    sep = '\t' if '\t' in header else ','
    sample_ids = [s.strip()[:15] for s in header.split(sep)[1:]]
    n_cols = len(sample_ids)
    print(f'Samples: {n_cols:,}')

    # Write header
    out.write('probe_id\t' + '\t'.join(sample_ids) + '\n')

    for line in fh:
        total_lines += 1
        if total_lines % 100_000 == 0:
            pct = found_count / max(len(TARGET_PROBES),1) * 100
            print(f'  {total_lines:,} lines | {found_count:,}/{len(TARGET_PROBES):,} probes ({pct:.0f}%)',
                  end='\r', flush=True)

        parts    = line.rstrip('\n').split(sep)
        probe_id = parts[0].strip().strip('"')

        if probe_id in TARGET_PROBES:
            vals    = parts[1:]
            trimmed = vals[:n_cols] + ['NA'] * (n_cols - len(vals[:n_cols]))
            out.write(probe_id + '\t' + '\t'.join(trimmed) + '\n')
            found_count += 1

        if found_count >= len(TARGET_PROBES):
            print(f'\n  All target probes found — stopping early!')
            break

print(f'\nScanned    : {total_lines:,} lines')
print(f'Probes written: {found_count:,}')
sz_mb = os.path.getsize(OUTPUT_FILE)/1e6
print(f'Output     : {OUTPUT_FILE}')
print(f'Size       : {sz_mb:.0f} MB')
print(f'Samples    : {n_cols:,}')
missing = len(TARGET_PROBES) - found_count
print(f'Not in TCGA: {missing:,} (TCGA QC exclusions — expected)')


## Step 4 — Verify and compare with full extraction

In [ ]:
import pandas as pd, os

print('=== Q-arm only extraction summary ===')
sz = os.path.getsize(OUTPUT_FILE)/1e6
print(f'Output file    : {OUTPUT_FILE}')
print(f'File size      : {sz:.0f} MB')
print(f'Probes written : {found_count:,}')
print(f'Samples        : {n_cols:,}')

# Compare with full extraction
FULL_FILE = OUTPUT_FILE.replace('q_only', 'full')
if os.path.exists(FULL_FILE):
    full_sz = os.path.getsize(FULL_FILE)/1e6
    print(f'\nComparison with full (p+q) extraction:')
    print(f'  Full file size : {full_sz:.0f} MB')
    print(f'  Q-only size    : {sz:.0f} MB  ({sz/full_sz*100:.1f}% of full)')
    print(f'  Full probes    : ~105,451')
    print(f'  Q-only probes  : {found_count:,}  ({found_count/105451*100:.1f}% of full)')

# Preview
df = pd.read_csv(OUTPUT_FILE, sep='\t', index_col=0, nrows=5)
print(f'\nPreview shape: {df.shape}')
print(df.iloc[:, :4])
print()
print('Next steps:')
print('1. Upload tcga_dmb_probes_q_only.tsv to Kaggle dataset tcga-pancan-methylation')
print('2. Run meth3dnet-v6-tcga-validation.ipynb with this file')
print('3. Compare Layer A/B/C results against the full p+q extraction')
